# 🧮 CSI as a SAT Hardness Predictor
## Millennium Lab — Empirical Validation Experiment

> **Hypothesis:** The Computational Symmetry Index (CSI = β₁ of the clause-variable interaction graph) predicts SAT instance hardness with signal *independent* of the standard clause/variable ratio α.

**Pipeline:**
1. Install dependencies
2. Download SATLIB benchmark instances
3. Parse CNF → build clause-variable interaction graph → compute CSI
4. Run MiniSat solver → log backtracks + solve time
5. Statistical analysis: Spearman correlation, partial correlation, phase transition plot
6. Interpret results

---

## Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install networkx python-sat pandas scipy matplotlib seaborn requests -q

# Install MiniSat system binary for backtrack counting
!apt-get install -y minisat -q

print('✅ All dependencies installed')

## Cell 2 — Imports & Configuration

In [ ]:
import os
import re
import time
import gzip
import subprocess
import requests
import tempfile
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from pysat.solvers import Minisat22
from pysat.formula import CNF
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# ── Visual config ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333333',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#888888',
    'ytick.color':      '#888888',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2a2a2a',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
    'axes.titlecolor':  '#ffffff',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

ACCENT   = '#00ff9d'   # neon green — primary signal
ACCENT2  = '#ff6b35'   # orange — baseline comparator
ACCENT3  = '#4ecdc4'   # teal — secondary
MUTED    = '#555555'

DATA_DIR = Path('/tmp/satlib')
DATA_DIR.mkdir(exist_ok=True)

print('✅ Imports complete | Theme configured')

## Cell 3 — Download SATLIB Benchmark Instances

We use three families:
- **uf50** — Random 3-SAT, 50 variables, satisfiable (easy-medium)
- **uf100** — Random 3-SAT, 100 variables, satisfiable (medium-hard)
- **uuf50** — Random 3-SAT, 50 variables, **unsatisfiable** (hard)

Each family has 1000 instances. We sample 200 per family for speed.

In [ ]:
SATLIB_URLS = {
    'uf50':  'https://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/RND3SAT/uf50-218.tar.gz',
    'uf100': 'https://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/RND3SAT/uf100-430.tar.gz',
    'uuf50': 'https://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/RND3SAT/uuf50-218.tar.gz',
}

def download_satlib(name, url, dest_dir):
    """Download and extract a SATLIB tar.gz archive."""
    family_dir = dest_dir / name
    if family_dir.exists() and any(family_dir.glob('*.cnf')):
        print(f'  {name}: already downloaded ({len(list(family_dir.glob("*.cnf")))} files)')
        return family_dir
    family_dir.mkdir(exist_ok=True)
    tar_path = dest_dir / f'{name}.tar.gz'
    print(f'  Downloading {name}...', end=' ')
    try:
        r = requests.get(url, timeout=30, stream=True)
        r.raise_for_status()
        with open(tar_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        !tar -xzf {tar_path} -C {family_dir} --strip-components=1 2>/dev/null || \
         tar -xzf {tar_path} -C {family_dir} 2>/dev/null
        # Move any nested cnf files up
        for cnf in family_dir.rglob('*.cnf'):
            if cnf.parent != family_dir:
                cnf.rename(family_dir / cnf.name)
        print(f'✅ {len(list(family_dir.glob("*.cnf")))} instances')
    except Exception as e:
        print(f'⚠️  Download failed: {e} — will use synthetic instances')
    return family_dir

print('📥 Downloading SATLIB benchmarks...')
family_dirs = {}
for name, url in SATLIB_URLS.items():
    family_dirs[name] = download_satlib(name, url, DATA_DIR)

## Cell 4 — Synthetic Instance Generator (Fallback)

If SATLIB download fails or for phase transition experiments, we generate random 3-SAT instances programmatically across a sweep of α values.

In [ ]:
def generate_random_3sat(n_vars, n_clauses, seed=None):
    """
    Generate a random 3-SAT instance.
    Returns list of clauses, each clause is a list of literals (±var).
    """
    rng = np.random.default_rng(seed)
    clauses = []
    for _ in range(n_clauses):
        vars_chosen = rng.choice(np.arange(1, n_vars + 1), size=3, replace=False)
        signs = rng.choice([-1, 1], size=3)
        clause = [int(s * v) for s, v in zip(signs, vars_chosen)]
        clauses.append(clause)
    return clauses

def write_cnf(clauses, n_vars, filepath):
    """Write clauses to DIMACS CNF format."""
    with open(filepath, 'w') as f:
        f.write(f'p cnf {n_vars} {len(clauses)}\n')
        for clause in clauses:
            f.write(' '.join(map(str, clause)) + ' 0\n')

def generate_phase_transition_instances(n_vars=50, alpha_range=None, n_per_alpha=20):
    """
    Generate instances sweeping alpha from 2.0 to 6.0.
    Returns list of (alpha, filepath) tuples.
    """
    if alpha_range is None:
        alpha_range = np.arange(2.0, 6.2, 0.2)
    pt_dir = DATA_DIR / 'phase_transition'
    pt_dir.mkdir(exist_ok=True)
    instances = []
    for alpha in alpha_range:
        n_clauses = int(alpha * n_vars)
        for i in range(n_per_alpha):
            fp = pt_dir / f'pt_n{n_vars}_a{alpha:.1f}_i{i}.cnf'
            clauses = generate_random_3sat(n_vars, n_clauses, seed=i * 1000 + int(alpha * 100))
            write_cnf(clauses, n_vars, fp)
            instances.append({'alpha': round(alpha, 2), 'filepath': fp})
    print(f'✅ Generated {len(instances)} phase-transition instances (α: 2.0 → 6.0)')
    return instances

# Generate phase transition instances (always — used for Experiment 3)
pt_instances = generate_phase_transition_instances(n_vars=50, n_per_alpha=30)
print(f'Phase transition dataset: {len(pt_instances)} instances across {len(set(i["alpha"] for i in pt_instances))} alpha values')

## Cell 5 — CNF Parser

In [ ]:
def parse_cnf(filepath):
    """
    Parse DIMACS CNF file.
    Returns: (n_vars, n_clauses_declared, clauses)
    clauses: list of lists of non-zero integers
    """
    clauses = []
    n_vars = 0
    n_clauses_declared = 0
    current_clause = []

    opener = gzip.open if str(filepath).endswith('.gz') else open
    with opener(filepath, 'rt', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('c'):
                continue
            if line.startswith('p cnf'):
                parts = line.split()
                n_vars = int(parts[2])
                n_clauses_declared = int(parts[3])
                continue
            # Parse literals
            for tok in line.split():
                try:
                    lit = int(tok)
                    if lit == 0:
                        if current_clause:
                            clauses.append(current_clause)
                            current_clause = []
                    else:
                        current_clause.append(lit)
                except ValueError:
                    continue
        if current_clause:  # Handle missing terminal 0
            clauses.append(current_clause)

    return n_vars, n_clauses_declared, clauses

# Quick test
test_clauses = generate_random_3sat(10, 43, seed=42)
test_fp = DATA_DIR / 'test.cnf'
write_cnf(test_clauses, 10, test_fp)
n_v, n_c, cls = parse_cnf(test_fp)
print(f'✅ Parser test: {n_v} vars, {n_c} clauses declared, {len(cls)} parsed')

## Cell 6 — CSI Computation

**CSI = β₁(G_φ)** where:
- G_φ is the clause-variable interaction graph (bipartite: clauses ↔ variables)
- β₁ = |E| − |V| + number of connected components (first Betti number)

High β₁ = more cycles = more entangled constraint structure = harder instance (hypothesis).

In [ ]:
def build_clause_variable_graph(clauses, n_vars):
    """
    Build bipartite clause-variable interaction graph.
    Clause nodes: 'c_{i}'
    Variable nodes: 'v_{j}'
    Edge: clause i mentions variable |lit|
    """
    G = nx.Graph()
    # Add variable nodes
    for v in range(1, n_vars + 1):
        G.add_node(f'v_{v}', bipartite=0)
    # Add clause nodes and edges
    for i, clause in enumerate(clauses):
        c_node = f'c_{i}'
        G.add_node(c_node, bipartite=1)
        for lit in clause:
            var = abs(lit)
            G.add_edge(c_node, f'v_{var}')
    return G

def compute_csi(G):
    """
    Compute CSI = β₁ = |E| - |V| + number_of_connected_components
    This is the first Betti number (cycle rank) of the graph.
    """
    E = G.number_of_edges()
    V = G.number_of_nodes()
    C = nx.number_connected_components(G)
    beta_1 = E - V + C
    return max(0, beta_1)  # β₁ is always ≥ 0

def compute_csi_normalized(G, n_vars, n_clauses):
    """CSI normalized by instance size for fair cross-family comparison."""
    raw = compute_csi(G)
    return raw / (n_vars + n_clauses) if (n_vars + n_clauses) > 0 else 0

def get_graph_features(G):
    """Additional graph features for richer analysis."""
    degrees = [d for _, d in G.degree()]
    return {
        'max_degree':    max(degrees) if degrees else 0,
        'mean_degree':   np.mean(degrees) if degrees else 0,
        'n_components':  nx.number_connected_components(G),
        'density':       nx.density(G),
    }

# Test CSI computation
G_test = build_clause_variable_graph(test_clauses, 10)
csi_test = compute_csi(G_test)
print(f'✅ CSI test: β₁ = {csi_test}')
print(f'   Graph: {G_test.number_of_nodes()} nodes, {G_test.number_of_edges()} edges, '
      f'{nx.number_connected_components(G_test)} components')

## Cell 7 — SAT Solver Integration

We use **PySAT's Minisat22** wrapper which gives us decision/propagation counts (proxy for backtracking intensity).

In [ ]:
def solve_instance(clauses, n_vars):
    """
    Solve a SAT instance using Minisat22 via PySAT.
    Returns: dict with solve stats.
    """
    cnf = CNF()
    cnf.nv = n_vars
    for clause in clauses:
        if clause:  # skip empty clauses
            cnf.append(clause)

    solver = Minisat22(bootstrap_with=cnf.clauses)

    t_start = time.perf_counter()
    result = solver.solve()
    t_end = time.perf_counter()

    accum = solver.accum_stats()
    solver.delete()

    return {
        'satisfiable':  result,
        'solve_ms':     round((t_end - t_start) * 1000, 4),
        'decisions':    accum.get('decisions', 0),      # ≈ backtrack count
        'propagations': accum.get('propagations', 0),
        'conflicts':    accum.get('conflicts', 0),      # direct backtrack count
    }

# Test solver
sol_test = solve_instance(test_clauses, 10)
print(f'✅ Solver test: SAT={sol_test["satisfiable"]}, '
      f'decisions={sol_test["decisions"]}, '
      f'conflicts={sol_test["conflicts"]}, '
      f'time={sol_test["solve_ms"]}ms')

## Cell 8 — Full Pipeline: Process All Instances

In [ ]:
def process_instance(filepath, family_label):
    """
    Full pipeline for one CNF instance:
    Parse → Graph → CSI → Solve → Return row dict
    """
    try:
        n_vars, _, clauses = parse_cnf(filepath)
        if not clauses or n_vars == 0:
            return None

        n_clauses = len(clauses)
        alpha = n_clauses / n_vars if n_vars > 0 else 0

        # Graph + CSI
        G = build_clause_variable_graph(clauses, n_vars)
        csi_raw   = compute_csi(G)
        csi_norm  = compute_csi_normalized(G, n_vars, n_clauses)
        gf        = get_graph_features(G)

        # Solve
        sol = solve_instance(clauses, n_vars)

        return {
            'filename':     Path(filepath).name,
            'family':       family_label,
            'n_vars':       n_vars,
            'n_clauses':    n_clauses,
            'alpha':        round(alpha, 3),
            'csi_raw':      csi_raw,
            'csi_norm':     round(csi_norm, 6),
            'max_degree':   gf['max_degree'],
            'mean_degree':  round(gf['mean_degree'], 3),
            'n_components': gf['n_components'],
            'density':      round(gf['density'], 6),
            'satisfiable':  sol['satisfiable'],
            'decisions':    sol['decisions'],
            'conflicts':    sol['conflicts'],
            'propagations': sol['propagations'],
            'solve_ms':     sol['solve_ms'],
        }
    except Exception as e:
        return None


def collect_instances(family_dirs, max_per_family=200):
    """Collect CNF files from all family directories."""
    all_files = []
    for name, dirpath in family_dirs.items():
        cnfs = sorted(dirpath.glob('*.cnf'))[:max_per_family]
        for fp in cnfs:
            all_files.append((fp, name))
    return all_files


# ── Run pipeline ──────────────────────────────────────────────
print('🔄 Processing SATLIB instances...')

satlib_files = collect_instances(family_dirs, max_per_family=200)

# Fallback: if no SATLIB files downloaded, generate synthetic ones
if len(satlib_files) < 10:
    print('⚠️  SATLIB files not found — generating synthetic instances')
    synth_dir = DATA_DIR / 'synthetic'
    synth_dir.mkdir(exist_ok=True)
    for family, (n_vars, alpha) in [('uf50', (50, 4.27)), ('uf100', (100, 4.27)), ('uuf50', (50, 4.27))]:
        fam_dir = synth_dir / family
        fam_dir.mkdir(exist_ok=True)
        family_dirs[family] = fam_dir
        for i in range(200):
            n_clauses = int(alpha * n_vars)
            clauses = generate_random_3sat(n_vars, n_clauses, seed=i)
            fp = fam_dir / f'{family}_{i:04d}.cnf'
            write_cnf(clauses, n_vars, fp)
    satlib_files = collect_instances(family_dirs, max_per_family=200)
    print(f'✅ Generated {len(satlib_files)} synthetic instances')

# Process
rows = []
for fp, family in tqdm(satlib_files, desc='Processing instances'):
    row = process_instance(fp, family)
    if row:
        rows.append(row)

df = pd.DataFrame(rows)
print(f'\n✅ Processed {len(df)} instances across {df["family"].nunique()} families')
df.head()

## Cell 9 — Process Phase Transition Dataset

In [ ]:
print('🔄 Processing phase transition instances...')
pt_rows = []
for item in tqdm(pt_instances, desc='Phase transition'):
    row = process_instance(item['filepath'], 'phase_transition')
    if row:
        row['alpha_sweep'] = item['alpha']
        pt_rows.append(row)

df_pt = pd.DataFrame(pt_rows)
print(f'✅ Phase transition dataset: {len(df_pt)} instances')
print(df_pt.groupby('alpha_sweep')['conflicts'].agg(['mean', 'std']).head(10))

## Cell 10 — Statistical Analysis

**Three tests:**
1. Spearman ρ: CSI vs conflicts (our key metric)
2. Spearman ρ: α vs conflicts (baseline)
3. Partial correlation: CSI vs conflicts, controlling for α

In [ ]:
def partial_correlation(df, x, y, control):
    """
    Compute partial correlation of x and y, controlling for `control`.
    Uses residuals method.
    """
    from scipy.stats import pearsonr
    # Residualize x on control
    slope_xc, intercept_xc, _, _, _ = stats.linregress(df[control], df[x])
    resid_x = df[x] - (slope_xc * df[control] + intercept_xc)
    # Residualize y on control
    slope_yc, intercept_yc, _, _, _ = stats.linregress(df[control], df[y])
    resid_y = df[y] - (slope_yc * df[control] + intercept_yc)
    # Correlation of residuals
    r, p = pearsonr(resid_x, resid_y)
    return r, p


# Filter: only use instances where solver actually did work
df_valid = df[df['conflicts'] >= 0].copy()
df_valid['log_conflicts'] = np.log1p(df_valid['conflicts'])
df_valid['log_solve_ms']  = np.log1p(df_valid['solve_ms'])

print('='*60)
print('STATISTICAL RESULTS')
print('='*60)

results = {}

for target_label, target_col in [('Conflicts (backtracks)', 'log_conflicts'), ('Solve time (ms)', 'log_solve_ms')]:
    print(f'\n── Target: {target_label} ──')

    # CSI raw
    rho_csi, p_csi = spearmanr(df_valid['csi_raw'], df_valid[target_col])
    print(f'  Spearman ρ (CSI_raw  vs {target_label[:12]}): {rho_csi:+.4f}  p={p_csi:.4e}')

    # CSI normalized
    rho_csin, p_csin = spearmanr(df_valid['csi_norm'], df_valid[target_col])
    print(f'  Spearman ρ (CSI_norm vs {target_label[:12]}): {rho_csin:+.4f}  p={p_csin:.4e}')

    # Alpha baseline
    rho_a, p_a = spearmanr(df_valid['alpha'], df_valid[target_col])
    print(f'  Spearman ρ (alpha    vs {target_label[:12]}): {rho_a:+.4f}  p={p_a:.4e}  ← baseline')

    # Partial correlation
    try:
        r_partial, p_partial = partial_correlation(df_valid, 'csi_norm', target_col, 'alpha')
        print(f'  Partial r  (CSI_norm | alpha-controlled):  {r_partial:+.4f}  p={p_partial:.4e}  ← KEY RESULT')
    except Exception as e:
        print(f'  Partial correlation failed: {e}')

    results[target_col] = {
        'rho_csi': rho_csi, 'p_csi': p_csi,
        'rho_alpha': rho_a, 'p_alpha': p_a,
    }

print('\n' + '='*60)
print('INTERPRETATION GUIDE')
print('='*60)
print('If |ρ_CSI| > |ρ_alpha|: CSI is a stronger predictor than α')
print('If partial r is significant (p < 0.05): CSI adds signal beyond α')
print('This is your key publishable finding.')

## Cell 11 — Visualization Suite

In [ ]:
fig = plt.figure(figsize=(20, 16), facecolor='#0d0d0d')
fig.suptitle('Computational Symmetry Index — Empirical Validation\nMillennium Lab · P vs NP Experiment',
             fontsize=15, color='#ffffff', fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

family_colors = {'uf50': ACCENT, 'uf100': ACCENT2, 'uuf50': ACCENT3, 'synthetic': '#aaaaaa'}

# ── Plot 1: CSI vs Conflicts (scatter, colored by family) ──
ax1 = fig.add_subplot(gs[0, 0:2])
for fam, grp in df_valid.groupby('family'):
    color = family_colors.get(fam, '#aaaaaa')
    ax1.scatter(grp['csi_raw'], np.log1p(grp['conflicts']),
                alpha=0.5, s=18, color=color, label=fam, edgecolors='none')
# Trend line
x_vals = df_valid['csi_raw'].values
y_vals = df_valid['log_conflicts'].values
if len(x_vals) > 2:
    m, b, r, p, se = stats.linregress(x_vals, y_vals)
    xr = np.linspace(x_vals.min(), x_vals.max(), 200)
    ax1.plot(xr, m * xr + b, color='#ffffff', linewidth=1.5, alpha=0.6, linestyle='--')
    rho, pv = spearmanr(x_vals, y_vals)
    ax1.text(0.98, 0.05, f'ρ = {rho:.3f}  p = {pv:.2e}', transform=ax1.transAxes,
             ha='right', va='bottom', color=ACCENT, fontsize=9)
ax1.set_xlabel('CSI (β₁ raw)')
ax1.set_ylabel('log(1 + conflicts)')
ax1.set_title('CSI vs Solver Conflicts')
ax1.legend(fontsize=8, framealpha=0.2)
ax1.grid(True, alpha=0.2)

# ── Plot 2: Alpha vs Conflicts (baseline comparison) ──
ax2 = fig.add_subplot(gs[0, 2])
ax2.scatter(df_valid['alpha'], df_valid['log_conflicts'],
            alpha=0.4, s=12, color=ACCENT2, edgecolors='none')
if len(df_valid) > 2:
    m2, b2, _, _, _ = stats.linregress(df_valid['alpha'], df_valid['log_conflicts'])
    xr2 = np.linspace(df_valid['alpha'].min(), df_valid['alpha'].max(), 100)
    ax2.plot(xr2, m2 * xr2 + b2, color='#ffffff', linewidth=1.2, alpha=0.5, linestyle='--')
    rho2, pv2 = spearmanr(df_valid['alpha'], df_valid['log_conflicts'])
    ax2.text(0.98, 0.05, f'ρ = {rho2:.3f}\np = {pv2:.2e}', transform=ax2.transAxes,
             ha='right', va='bottom', color=ACCENT2, fontsize=8)
ax2.set_xlabel('α (clauses/vars)')
ax2.set_ylabel('log(1 + conflicts)')
ax2.set_title('α vs Conflicts (Baseline)')
ax2.grid(True, alpha=0.2)

# ── Plot 3: CSI distribution by family ──
ax3 = fig.add_subplot(gs[1, 0])
for fam, grp in df_valid.groupby('family'):
    color = family_colors.get(fam, '#aaaaaa')
    ax3.hist(grp['csi_raw'], bins=30, alpha=0.6, color=color, label=fam, edgecolor='none')
ax3.set_xlabel('CSI (β₁ raw)')
ax3.set_ylabel('Count')
ax3.set_title('CSI Distribution by Family')
ax3.legend(fontsize=8, framealpha=0.2)
ax3.grid(True, alpha=0.2)

# ── Plot 4: CSI vs Solve Time ──
ax4 = fig.add_subplot(gs[1, 1])
ax4.scatter(df_valid['csi_norm'], np.log1p(df_valid['solve_ms']),
            alpha=0.45, s=14, color=ACCENT3, edgecolors='none')
rho4, pv4 = spearmanr(df_valid['csi_norm'], np.log1p(df_valid['solve_ms']))
ax4.text(0.98, 0.05, f'ρ = {rho4:.3f}\np = {pv4:.2e}', transform=ax4.transAxes,
         ha='right', va='bottom', color=ACCENT3, fontsize=8)
ax4.set_xlabel('CSI normalized')
ax4.set_ylabel('log(1 + solve_ms)')
ax4.set_title('CSI_norm vs Solve Time')
ax4.grid(True, alpha=0.2)

# ── Plot 5: Satisfiable vs Unsatisfiable CSI boxplot ──
ax5 = fig.add_subplot(gs[1, 2])
sat_data   = df_valid[df_valid['satisfiable'] == True]['csi_raw'].dropna()
unsat_data = df_valid[df_valid['satisfiable'] == False]['csi_raw'].dropna()
bp = ax5.boxplot([sat_data, unsat_data], patch_artist=True,
                 medianprops={'color': '#ffffff', 'linewidth': 2})
bp['boxes'][0].set_facecolor(ACCENT + '66')
if len(bp['boxes']) > 1:
    bp['boxes'][1].set_facecolor(ACCENT2 + '66')
ax5.set_xticklabels(['SAT', 'UNSAT'])
ax5.set_ylabel('CSI (β₁ raw)')
ax5.set_title('CSI: SAT vs UNSAT')
ax5.grid(True, alpha=0.2)
if len(sat_data) > 0 and len(unsat_data) > 0:
    stat, pval = stats.mannwhitneyu(sat_data, unsat_data, alternative='two-sided')
    ax5.text(0.5, 0.95, f'Mann-Whitney p={pval:.3e}', transform=ax5.transAxes,
             ha='center', va='top', color='#cccccc', fontsize=8)

# ── Plot 6: Phase Transition — CSI and Conflicts vs Alpha ──
ax6 = fig.add_subplot(gs[2, 0:2])
if len(df_pt) > 0:
    df_pt['log_conflicts'] = np.log1p(df_pt['conflicts'])
    pt_agg = df_pt.groupby('alpha_sweep').agg(
        csi_mean=('csi_raw', 'mean'),
        csi_std=('csi_raw', 'std'),
        conf_mean=('log_conflicts', 'mean'),
        conf_std=('log_conflicts', 'std'),
    ).reset_index()

    ax6b = ax6.twinx()

    ax6.plot(pt_agg['alpha_sweep'], pt_agg['csi_mean'], color=ACCENT,
             linewidth=2.5, label='CSI (β₁)', marker='o', markersize=4)
    ax6.fill_between(pt_agg['alpha_sweep'],
                     pt_agg['csi_mean'] - pt_agg['csi_std'].fillna(0),
                     pt_agg['csi_mean'] + pt_agg['csi_std'].fillna(0),
                     alpha=0.15, color=ACCENT)

    ax6b.plot(pt_agg['alpha_sweep'], pt_agg['conf_mean'], color=ACCENT2,
              linewidth=2, linestyle='--', label='log(conflicts)', marker='s', markersize=3)

    ax6.axvline(x=4.27, color='#ffffff', linewidth=1, linestyle=':', alpha=0.7)
    ax6.text(4.32, ax6.get_ylim()[1] * 0.9 if ax6.get_ylim()[1] > 0 else 1,
             'α≈4.27\n(phase transition)', color='#ffffff', fontsize=8, alpha=0.8)

    ax6.set_xlabel('α (clause/variable ratio)')
    ax6.set_ylabel('Mean CSI (β₁)', color=ACCENT)
    ax6b.set_ylabel('Mean log(1+conflicts)', color=ACCENT2)
    ax6.set_title('Phase Transition: CSI and Hardness vs α')
    lines1, labels1 = ax6.get_legend_handles_labels()
    lines2, labels2 = ax6b.get_legend_handles_labels()
    ax6.legend(lines1 + lines2, labels1 + labels2, fontsize=8, framealpha=0.2, loc='upper left')
    ax6.grid(True, alpha=0.2)

# ── Plot 7: Correlation heatmap ──
ax7 = fig.add_subplot(gs[2, 2])
corr_cols = ['csi_raw', 'csi_norm', 'alpha', 'max_degree', 'n_components', 'conflicts', 'solve_ms']
corr_cols_present = [c for c in corr_cols if c in df_valid.columns]
corr_matrix = df_valid[corr_cols_present].corr(method='spearman')
labels_short = [c.replace('_', '\n') for c in corr_cols_present]
im = ax7.imshow(corr_matrix.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax7.set_xticks(range(len(corr_cols_present)))
ax7.set_yticks(range(len(corr_cols_present)))
ax7.set_xticklabels(labels_short, fontsize=7, rotation=45, ha='right')
ax7.set_yticklabels(labels_short, fontsize=7)
for i in range(len(corr_cols_present)):
    for j in range(len(corr_cols_present)):
        ax7.text(j, i, f'{corr_matrix.values[i,j]:.2f}', ha='center', va='center',
                 fontsize=6, color='black' if abs(corr_matrix.values[i,j]) < 0.5 else 'white')
ax7.set_title('Spearman Correlation Heatmap')
plt.colorbar(im, ax=ax7, shrink=0.8)

plt.savefig('/tmp/csi_results.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('✅ Visualization saved to /tmp/csi_results.png')

## Cell 12 — Results Summary & Paper-Ready Statistics

In [ ]:
print('=' * 65)
print('MILLENNIUM LAB — CSI EXPERIMENT RESULTS SUMMARY')
print('=' * 65)

print(f'\nDataset: {len(df_valid)} instances across {df_valid["family"].nunique()} families')
print(df_valid.groupby('family')[['n_vars', 'n_clauses', 'csi_raw', 'conflicts']].mean().round(2).to_string())

print('\n── Primary Correlations (vs log_conflicts) ──')
for col, label in [('csi_raw', 'CSI raw  '), ('csi_norm', 'CSI norm '), ('alpha', 'alpha    '), ('max_degree', 'max_degree')]:
    if col in df_valid.columns:
        rho, p = spearmanr(df_valid[col], df_valid['log_conflicts'])
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f'  {label}: ρ = {rho:+.4f}  p = {p:.3e}  {sig}')

print('\n── Partial Correlation: CSI_norm | alpha ──')
try:
    r_p, p_p = partial_correlation(df_valid, 'csi_norm', 'log_conflicts', 'alpha')
    sig = '***' if p_p < 0.001 else ('**' if p_p < 0.01 else ('*' if p_p < 0.05 else 'ns (no independent signal)'))
    print(f'  r_partial = {r_p:+.4f}  p = {p_p:.3e}  {sig}')
    if p_p < 0.05:
        print('  ✅ CSI captures hardness signal BEYOND what α explains')
    else:
        print('  ⚠️  CSI does not add independent signal over α at p<0.05')
        print('     → Consider refining CSI definition (e.g. weighted Betti number)')
except Exception as e:
    print(f'  Could not compute: {e}')

print('\n── Phase Transition ──')
if len(df_pt) > 0:
    pt_agg2 = df_pt.groupby('alpha_sweep')['csi_raw'].mean()
    peak_alpha = pt_agg2.idxmax()
    print(f'  CSI peaks at α = {peak_alpha:.2f}  (3-SAT phase transition ≈ 4.27)')
    if abs(peak_alpha - 4.27) < 0.3:
        print('  ✅ CSI peak aligns with phase transition — topological signal confirmed')
    else:
        print(f'  ⚠️  CSI peak at α={peak_alpha:.2f} does not closely track phase transition')

print('\n── Paper-Ready Sentence ──')
rho_csi_v, p_csi_v = spearmanr(df_valid['csi_raw'], df_valid['log_conflicts'])
rho_a_v, p_a_v     = spearmanr(df_valid['alpha'],   df_valid['log_conflicts'])
print(f'''  "Spearman ρ between CSI and solver conflict count = {rho_csi_v:.3f} (p={p_csi_v:.2e}),
  compared to ρ = {rho_a_v:.3f} (p={p_a_v:.2e}) for clause-density α.
  CSI peaks at α ≈ {peak_alpha if len(df_pt) > 0 else "N/A":.2f}, coinciding with the 3-SAT phase transition."''')

print('\n' + '=' * 65)
print('Next steps if partial correlation is significant:')
print('  1. Run on SATLIB uf250 (larger instances) for robustness')
print('  2. Compare against treewidth and backdoor size predictors')
print('  3. Write up as: arXiv cs.CC or JSAT submission')
print('=' * 65)

## Cell 13 — Export Results

In [ ]:
# Save full results dataframe
df_valid.to_csv('/tmp/csi_results_full.csv', index=False)
df_pt.to_csv('/tmp/csi_phase_transition.csv', index=False)

print('✅ Exported:')
print('  /tmp/csi_results_full.csv     — full instance-level results')
print('  /tmp/csi_phase_transition.csv — phase transition sweep')
print('  /tmp/csi_results.png          — visualization suite')
print()
print('To download from Colab: Files → /tmp → right-click → Download')